# Price-Trend CNN with VIX Gate

This notebook is the Jupyter companion for `price_trend_vix_cnn.py`. It runs the embedded CPU-friendly price-trend CNN, applies a VIX exposure gate such as `gate_16_21`, and writes visual diagnostics showing where the signal works or fails across VIX levels.

Gate syntax: `gate_{lower}_{upper}` trades outside the middle VIX band. Use `nan` for an open side, for example `gate_nan_21` trades only when VIX is above 21.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import pandas as pd

import price_trend_vix_cnn as m

print('module:', m.__file__)

## Parameters

`PATTERN='sp00_5MA'` is accepted by the wrapper and falls back to `SP500_5MA` if needed. The notebook trains end-to-end from the chart-image pickle every time; it does not read any existing `predictions.csv` cache.

For the manager checks, set `DROP_TRAIN_ABS_RET` to `0.10`, `0.15`, or `0.20`, or set `TRAIN_VIX_LT = 20.0`. These filters only mask training labels before fitting; the 2015-2019 test labels are left untouched.

In [ ]:
DATA_DIR = Path('.')
PATTERN = 'sp00_5MA'
GATE = 'gate_16_21'

TEST_START = '2015'
TEST_END = '2019-12-31'
TRAIN_YEARS = 8
CPU_THREADS = 'auto'

DROP_TRAIN_ABS_RET = None  # examples: 0.10, 0.15, 0.20
TRAIN_VIX_LT = None        # example: 20.0

filter_bits = []
if DROP_TRAIN_ABS_RET is not None:
    filter_bits.append(f'drop_abs{DROP_TRAIN_ABS_RET:g}')
if TRAIN_VIX_LT is not None:
    filter_bits.append(f'train_vix_lt{TRAIN_VIX_LT:g}')
FILTER_TAG = ('_' + '_'.join(filter_bits).replace('.', 'p')) if filter_bits else ''

OUT_DIR = DATA_DIR / f'results_price_trend_vix_{GATE}{FILTER_TAG}'
SOURCE_OUT_DIR = DATA_DIR / f'results_price_trend_vix_{GATE}{FILTER_TAG}_raw'

DATA_DIR, PATTERN, GATE, OUT_DIR

## Run

This cell returns a dictionary with `summary`, raw `predictions`, period diagnostics, VIX bucket summary, and the output folder. The default uses CPU thread settings and does not save model weights.

In [ ]:
res = m.run(
    data_dir=DATA_DIR,
    pattern=PATTERN,
    gate=GATE,
    horizon=5,
    train_years=TRAIN_YEARS,
    test_start=TEST_START,
    test_end=TEST_END,
    cpu_threads=CPU_THREADS,
    drop_train_abs_ret=DROP_TRAIN_ABS_RET,
    train_vix_lt=TRAIN_VIX_LT,
    out_dir=OUT_DIR,
    source_out_dir=SOURCE_OUT_DIR,
    save_model=False,
    make_plots=True,
    verbose=True,
)

res['out_dir']

## Summary

In [ ]:
pd.DataFrame([res['summary']]).T.rename(columns={0: 'value'})

## VIX Diagnostics

In [ ]:
res['vix_bucket_summary']

In [ ]:
m.display_results(res)

## Inspect Periods

These rows show where the model works or fails by rebalance week, VIX, active gate status, information coefficient, and long-short return.

In [ ]:
periods = res['period_diagnostics'].copy()
periods.sort_values('long_short').head(20)[[
    'per', 'rebalance_date', 'vix', 'active', 'vix_regime',
    'ic_spearman', 'long_short', 'long_only', 'top_hit', 'bottom_hit', 'wrong_tail'
]]

In [ ]:
periods.sort_values('long_short', ascending=False).head(20)[[
    'per', 'rebalance_date', 'vix', 'active', 'vix_regime',
    'ic_spearman', 'long_short', 'long_only', 'top_hit', 'bottom_hit', 'wrong_tail'
]]

## Try Another Gate

Change `NEXT_GATE` and rerun this cell to compare gates without editing the main parameters above. This reruns the raw CNN as well, so the comparison is fully end-to-end from the same chart-image pickle.

In [ ]:
NEXT_GATE = 'gate_15_20'

res2 = m.run(
    data_dir=DATA_DIR,
    pattern=PATTERN,
    gate=NEXT_GATE,
    horizon=5,
    train_years=TRAIN_YEARS,
    test_start=TEST_START,
    test_end=TEST_END,
    cpu_threads=CPU_THREADS,
    drop_train_abs_ret=DROP_TRAIN_ABS_RET,
    train_vix_lt=TRAIN_VIX_LT,
    out_dir=DATA_DIR / f'results_price_trend_vix_{NEXT_GATE}{FILTER_TAG}',
    source_out_dir=DATA_DIR / f'results_price_trend_vix_{NEXT_GATE}{FILTER_TAG}_raw',
    save_model=False,
    make_plots=True,
    verbose=False,
)

pd.DataFrame([res['summary'], res2['summary']])[[
    'gate', 'active_periods', 'ls_sharpe', 'ls_cagr', 'lo_sharpe',
    'active_ls_mean', 'inactive_ls_mean', 'active_ic_mean', 'inactive_ic_mean'
]]